# Week 11, Day 3: CSV files

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w011d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
line = 'Ada,"Lagos, Nigeria",93'
print(line.split(","))
print(len(line.split(",")))

These are the files this page's programs use.

In [ ]:
# Run this cell first: it creates the files used below.
with open('students.csv', 'w', encoding='utf-8') as f:
    f.write('name,city,maths,english\nAda,"Lagos, Nigeria",93,71\nBo,Accra,41,88\nCy,"Nairobi, Kenya",78,64\nDi,Kigali,55,90\n')
with open('sales.csv', 'w', encoding='utf-8') as f:
    f.write('date,item,qty,price\n2026-10-01,rice,2,8500\n2026-10-01,beans,1,3200\n2026-10-02,rice,1,8500\n2026-10-03,oil,3,2100\n2026-10-03,beans,2,3200\n')

> **Tip.** Open `students.csv` in the box above and compare it with what the programs below print. Then try saving any spreadsheet as CSV on your own computer and opening it in a text editor: it's the same format.

In [ ]:
import csv
with open("students.csv", newline="") as f:
    for row in csv.reader(f):
        print(row)

In [ ]:
import csv
with open("students.csv", newline="") as f:
    reader = csv.reader(f)
    header = next(reader)
    rows = list(reader)
print(header)
print(len(rows), "students")
print(rows[2][1])

In [ ]:
import csv
total = 0
with open("sales.csv", newline="") as f:
    reader = csv.reader(f)
    next(reader)
    for date, item, qty, price in reader:
        total += int(qty) * int(price)
print(f"Total sales: {total:,} naira")

In [ ]:
import csv
with open("students.csv", newline="") as f:
    for record in csv.DictReader(f):
        print(record["name"], record["city"], record["maths"])

In [ ]:
import csv
with open("students.csv", newline="") as f:
    students = list(csv.DictReader(f))
for s in students:
    s["maths"] = int(s["maths"])
    s["english"] = int(s["english"])
best = max(students, key=lambda s: s["maths"])
print("Best at maths:", best["name"])
print("Average English:", sum(s["english"] for s in students) / len(students))

In [ ]:
import csv
from collections import defaultdict
qty_by_item = defaultdict(int)
with open("sales.csv", newline="") as f:
    for sale in csv.DictReader(f):
        qty_by_item[sale["item"]] += int(sale["qty"])
print(dict(qty_by_item))

In [ ]:
import csv
rows = [["name", "note"], ["Ada", "top, again"], ["Bo", 'said "hi"']]
with open("notes.csv", "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(rows)
with open("notes.csv") as f:
    print(f.read())

In [ ]:
import csv
people = [{"name": "Ada", "score": 93}, {"name": "Bo", "score": 41}]
with open("people.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "score"])
    writer.writeheader()
    writer.writerows(people)
with open("people.csv") as f:
    print(f.read())

> **Common mistake.** Without `newline=""` when writing, some systems (Windows in particular) put an extra blank line between the rows. Use it every time you open a CSV file, for reading or writing.

In [ ]:
import csv
with open("students.csv", newline="") as f:
    students = list(csv.DictReader(f))
for s in students:
    s["average"] = (int(s["maths"]) + int(s["english"])) / 2
with open("averages.csv", "w", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["name", "city", "maths", "english", "average"])
    writer.writeheader()
    writer.writerows(students)
with open("averages.csv") as f:
    print(f.read())

### Exercise: How many?

Write `count_records(name)` that returns the number of data rows in a CSV file, not counting the header.

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several files\"\n    assert count_records(\"students.csv\") == 4\n    assert count_records(\"sales.csv\") == 5\n    write_file(\"h.csv\", \"a,b\\n\")\n    assert count_records(\"h.csv\") == 0")

### Exercise: One column

Write `column(name, heading)` that returns the values of one column of a CSV file as a list of strings, using `DictReader`.

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several columns\"\n    assert column(\"students.csv\", \"city\") == [\"Lagos, Nigeria\", \"Accra\", \"Nairobi, Kenya\", \"Kigali\"]\n    assert column(\"sales.csv\", \"item\") == [\"rice\", \"beans\", \"rice\", \"oil\", \"beans\"]\n    assert \"DictReader\" in source")

### Exercise: Revenue per day

Write `daily_revenue(name)` that reads a sales file like `sales.csv` and returns a dictionary from date to that day's total (quantity times price).

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"sales\"\n    assert daily_revenue(\"sales.csv\") == {\"2026-10-01\": 20200, \"2026-10-02\": 8500, \"2026-10-03\": 12700}")

### Exercise: Write the pass list

Read `students.csv` and write `passed.csv` with the columns `name` and `maths`, holding only students with 50 or more in maths, in the same order. Use `csv.writer`.

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_file():\n    \"the pass list\"\n    run()\n    assert norm(read_file(\"passed.csv\")) == \"name,maths\\nAda,93\\nCy,78\\nDi,55\"")

### Exercise: Dictionaries to CSV

Write `save_records(records, name)` that writes a list of dictionaries to a CSV file with a header. The columns are the keys of the first record, in its order. An empty list writes an empty file.

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"several\"\n    save_records([{\"x\": 1, \"y\": \"a, b\"}, {\"x\": 2, \"y\": \"c\"}], \"r.csv\")\n    assert norm(read_file(\"r.csv\")) == 'x,y\\n1,\"a, b\"\\n2,c'\n    save_records([], \"e.csv\")\n    assert read_file(\"e.csv\") == \"\"")

### Exercise: A semicolon file

`prices.csv` uses semicolons and a decimal comma, like `rice;8500,50`. Write `load_prices(name)` that returns a dictionary from item to price as a float. There's no header.

In [ ]:
# Run this cell first: it creates the files used below.
with open('prices.csv', 'w', encoding='utf-8') as f:
    f.write('rice;8500,50\nbeans;3200\n"oil; palm";2100,75\n')

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"the prices\"\n    assert load_prices(\"prices.csv\") == {\"rice\": 8500.5, \"beans\": 3200.0, \"oil; palm\": 2100.75}")

### Bug hunt: Too many fields

This should print each student's city, but `Lagos, Nigeria` comes out as `"Lagos`. Fix it with the `csv` module.

In [ ]:
with open("students.csv") as f:
    next(f)
    for line in f:
        fields = line.strip().split(",")
        print(fields[1])

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the cities\"\n    expect_output(\"Lagos, Nigeria\\nAccra\\nNairobi, Kenya\\nKigali\")")

### Bug hunt: String sums

This should print the total quantity sold, `9`, but it crashes. Fix it.

In [ ]:
import csv
total = 0
with open("sales.csv", newline="") as f:
    for sale in csv.DictReader(f):
        total += sale["qty"]
print(total)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"nine\"\n    expect_output(\"9\")")

### Bug hunt: The header counts as a student

This should print the highest maths mark, `93`, but crashes with `ValueError: invalid literal for int() with base 10: 'maths'`. Fix it.

In [ ]:
import csv
with open("students.csv", newline="") as f:
    marks = [int(row[2]) for row in csv.reader(f)]
print(max(marks))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"93\"\n    expect_output(\"93\")")

### Puzzle: Names in capitals

Arrange the lines to write `upper.csv`, a copy of `students.csv` with the names in capitals. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them

Work these out without running them. `csv.reader` can read from any list of lines, not just a file.

In [ ]:
import csv
a = ___   # next(csv.reader(['x,"y,z",1']))
b = ___   # len(next(csv.reader(['a,,b'])))
c = ___   # next(csv.reader(['"say ""hi"""']))[0]
d = ___   # type(next(csv.reader(['42']))[0])     (the type itself)

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each value\"\n    assert a == [\"x\", \"y,z\", \"1\"]\n    assert b == 3\n    assert c == 'say \"hi\"'\n    assert d is str")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a pivot table

Spreadsheets have **pivot tables**, which summarise a long table as a grid. Write `pivot(source, target)` that reads a sales file like `sales.csv` and writes `target`, a CSV with one row per **item** (sorted A to Z) and one column per **date** (sorted), holding the quantity of that item sold on that date (0 if none), plus a final `total` column. The first column is headed `item`.

For `sales.csv` the target should be:

```
item,2026-10-01,2026-10-02,2026-10-03,total
beans,1,0,2,3
oil,0,0,3,3
rice,2,1,0,3
```

In [ ]:
import csv

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_example():\n    \"sales.csv\"\n    pivot(\"sales.csv\", \"pivot.csv\")\n    assert norm(read_file(\"pivot.csv\")) == \"item,2026-10-01,2026-10-02,2026-10-03,total\\nbeans,1,0,2,3\\noil,0,0,3,3\\nrice,2,1,0,3\"\ndef test_other():\n    \"another file\"\n    write_file(\"s2.csv\", \"date,item,qty,price\\nd2,yam,4,1\\nd1,yam,1,1\\nd2,yam,1,1\\nd1,egg,6,1\\n\")\n    pivot(\"s2.csv\", \"p2.csv\")\n    assert norm(read_file(\"p2.csv\")) == \"item,d1,d2,total\\negg,6,0,6\\nyam,1,5,6\"")

## Warm-up

Three fields, but `split` found four, because one field contains a comma. The quotes say "this comma is part of the text", and `split` doesn't know that. The `csv` module does.

## The CSV format

A CSV file is a text file holding a table:

- each **line** is one row;
- fields are separated by **commas**;
- the first row is usually a **header** with the column names;
- a field containing a comma, a quote or a newline is wrapped in **double quotes**, and a quote inside it is doubled: `"She said ""hi"""`.

Everything in a CSV file is **text**. The file doesn't say that `93` is a number; your program decides that.

## Reading rows

`csv.reader` takes an open file and gives you each row as a **list of strings**, with the quoting handled:

`"Lagos, Nigeria"` came through as one field, without its quotes. Note `newline=""` in `open`: the `csv` module handles line endings itself, and this stops Python changing them first. Always use it for CSV files.

### The header row

The first row is the header. Take it with `next`, which gets the next item from the reader, and loop over the rest:

### Converting fields

Every field is a string, so convert the numbers yourself as you read:

Unpacking each row into named variables (`for date, item, qty, price in reader`) makes the code far easier to read than `row[2]` and `row[3]`.

## Reading records

`csv.DictReader` reads the header for you, and gives each row as a **dictionary** from column name to value:

This is usually the clearest way to read CSV: `record["maths"]` says what it means, and the code keeps working if someone adds a column or changes the order of the columns.

Read the whole file into a list of dictionaries, the shape from Week 10, and every pattern you know applies:

## Writing

### Rows with csv.writer

`csv.writer` writes lists as rows, adding commas and quotes wherever they're needed:

`writerow(row)` writes one row; `writerows(rows)` writes them all. Numbers are turned into text for you. Check the output: the field with a comma was quoted, and the quotes inside `said "hi"` were doubled. Writing CSV by joining strings with commas gets this wrong.

### Records with csv.DictWriter

`csv.DictWriter` writes dictionaries. You tell it the column names, in order, with `fieldnames`, and write the header with `writeheader()`:

## A complete round trip

Read a table, add a column, and write it out. This is one of the most common small programs anyone writes:

### Other separators

Some "CSV" files use semicolons or tabs instead of commas (common in countries where the comma is the decimal point). Tell the reader and writer with `delimiter`:

```
csv.reader(f, delimiter=";")
csv.writer(f, delimiter="\t")
```

## Practice

## Quiz

## Daily challenge

## Before you go

Find a CSV file of your own: download one from a public data site, or save a spreadsheet as CSV. Open it in a text editor, then read it with `DictReader` and print one column. Tomorrow: what to do when things go wrong, with `try` and `except`.